In [4]:
import pandas as pd

df = pd.read_csv('play_info.csv', encoding='cp932')  # 既知の文字化け対策

# チームごとにユニークな球種数を算出
team_pitchtype = df.groupby('pitcher_team_name')['pitch_type_name'].nunique().reset_index()
team_pitchtype.columns = ['team_name', 'num_pitch_types']
print(team_pitchtype)

/var/folders/wb/_p_lg6k167q3590thf3r_xyr0000gn/T/ipykernel_34261/1252197569.py:3: DtypeWarning: Columns (125,141) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('play_info.csv', encoding='cp932')  # 既知の文字化け対策


   team_name  num_pitch_types
0       DeNA                8
1      オリックス                9
2     ソフトバンク                8
3       ヤクルト                8
4        ロッテ                8
5         中日                8
6         巨人                8
7         広島                9
8       日本ハム                9
9         楽天                9
10        西武                9
11        阪神                9


In [6]:
import pandas as pd

# play_info.csv からデータ読込
df = pd.read_csv('play_info.csv', encoding='cp932')

# ゲーム終了時（最終イニング、最終打席）だけ抽出
# game_idごとに最大seqno_9（または最大inning, pa_of_inning）で抽出
last_play = df.sort_values(['game_id', 'seqno_9']).groupby('game_id').tail(1)

# 勝ち負け判定（得点部分のカラム名要確認: post_home_team_score, post_away_team_score）
# ホーム勝ち
last_play['home_win'] = last_play['post_home_team_score'] > last_play['post_away_team_score']
last_play['away_win'] = last_play['post_home_team_score'] < last_play['post_away_team_score']

# 集計
home_results = last_play.groupby('home_team_name').agg(
    games=('game_id','count'),
    wins=('home_win','sum')
).reset_index()
away_results = last_play.groupby('away_team_name').agg(
    games=('game_id','count'),
    wins=('away_win','sum')
).reset_index()

# 勝ち数合算
team_results = (
    pd.concat([
        home_results.rename(columns={'home_team_name':'team_name'}),
        away_results.rename(columns={'away_team_name':'team_name'})
    ])
    .groupby('team_name')
    .agg({'games':'sum','wins':'sum'})
    .reset_index()
)
team_results['win_rate'] = team_results['wins'] / team_results['games']

# カラムを合わせる
team_stats = team_results[['team_name','win_rate']]

# ---- 【ここから合体】 ----

# 球種多様性とのマージ
merged = pd.merge(team_pitchtype, team_stats, on='team_name')
print(merged)

/var/folders/wb/_p_lg6k167q3590thf3r_xyr0000gn/T/ipykernel_34261/299361897.py:4: DtypeWarning: Columns (125,141) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('play_info.csv', encoding='cp932')


   team_name  num_pitch_types  win_rate
0       DeNA                8  0.496503
1      オリックス                9  0.517483
2     ソフトバンク                8  0.608392
3       ヤクルト                8  0.398601
4        ロッテ                8  0.391608
5         中日                8  0.440559
6         巨人                8  0.489510
7         広島                9  0.412587
8       日本ハム                9  0.580420
9         楽天                9  0.468531
10        西武                9  0.440559
11        阪神                9  0.594406


In [7]:
import pandas as pd
import sqlite3

# ここでは既にDataFrameが手元にある前提（例：df = merged）
df = merged

# SQLite DBを作成して接続
con = sqlite3.connect('team_pitch_stats.db')

# データを書き込む（新規作成 or 既存テーブルを置き換え）
df.to_sql('team_pitch_summary', con, if_exists='replace', index=False)

con.close()

In [9]:
import pandas as pd

# データ読み込み
df = pd.read_csv('play_info.csv', encoding='cp932', low_memory=False)

# 例：team_statsにチーム名と勝率が入っているとします
# 無い場合はあなたの手元のチーム勝率DataFrame 例:
# team_stats = pd.DataFrame({
#    'team_name': [...],
#    'win_rate': [...]
# })

# 各チーム・球種の投球数
pitch_counts = (
    df.groupby(['pitcher_team_name', 'pitch_type_name'])
      .size()
      .reset_index(name='num_pitches')
)

# 各チーム全体の投球数
team_totals = (
    df.groupby('pitcher_team_name')
      .size()
      .reset_index(name='total_pitches')
)

# 合体して割合列を追加
pitch_stats = pd.merge(pitch_counts, team_totals, on='pitcher_team_name')
pitch_stats['pitch_ratio'] = pitch_stats['num_pitches'] / pitch_stats['total_pitches']

# チーム名のカラム名を合わせる
# あなたの勝率データに合わせて（たとえば 'team_name' or 'pitcher_team_name'）ください
pitch_stats = pitch_stats.rename(columns={'pitcher_team_name': 'team_name'})
# 勝率データと結合
result = pd.merge(pitch_stats, team_stats, on='team_name')

# 出力イメージ（見やすく並べ替え）
result = result[['team_name', 'pitch_type_name', 'pitch_ratio', 'win_rate', 'num_pitches']]

print(result.head())

# ---- CSVファイルとして保存 ----
result.to_csv('team_pitch_type_ratio_winrate.csv', index=False, encoding='utf-8-sig')

# ---- SQLiteデータベースに保存 ----
import sqlite3
con = sqlite3.connect('team_pitch_types.db')
result.to_sql('team_pitch_type_stats', con, if_exists='replace', index=False)
con.close()

  team_name pitch_type_name  pitch_ratio  win_rate  num_pitches
0      DeNA          カットボール     0.090618  0.496503         1895
1      DeNA             カーブ     0.074503  0.496503         1558
2      DeNA            シュート     0.078806  0.496503         1648
3      DeNA            シンカー     0.024579  0.496503          514
4      DeNA           ストレート     0.379830  0.496503         7943


In [10]:
result.to_csv('all_teams_pitch_ratio_winrate.csv', index=False, encoding='utf-8-sig')

In [11]:
import pandas as pd

# --- ① 最大割合の球種（最も多い） ---
max_pitch = result.loc[result.groupby('team_name')['pitch_ratio'].idxmax()] \
    .reset_index(drop=True)
max_pitch = max_pitch.rename(columns={
    'pitch_type_name': 'most_pitch_type',
    'pitch_ratio': 'most_pitch_ratio'
})

# --- ② 最小割合の球種（最も少ない） ---
min_pitch = result.loc[result.groupby('team_name')['pitch_ratio'].idxmin()] \
    .reset_index(drop=True)
min_pitch = min_pitch.rename(columns={
    'pitch_type_name': 'least_pitch_type',
    'pitch_ratio': 'least_pitch_ratio'
})

# --- ③ チームごとに結合し、勝率は一つで代表させる ---
summary = pd.merge(
    max_pitch[['team_name', 'most_pitch_type', 'most_pitch_ratio', 'win_rate']],
    min_pitch[['team_name', 'least_pitch_type', 'least_pitch_ratio']],
    on='team_name'
)

# --- ④ 出力 ---
print(summary)

# --- ⑤ CSV等で保存 ---
summary.to_csv('each_team_pitch_summary.csv', index=False, encoding='utf-8-sig')

   team_name most_pitch_type  most_pitch_ratio  win_rate least_pitch_type  \
0       DeNA           ストレート          0.379830  0.496503             シンカー   
1      オリックス           ストレート          0.342645  0.517483              特殊球   
2     ソフトバンク           ストレート          0.407382  0.608392             シンカー   
3       ヤクルト           ストレート          0.386334  0.398601             シンカー   
4        ロッテ           ストレート          0.483901  0.391608             シンカー   
5         中日           ストレート          0.403040  0.440559             シンカー   
6         巨人           ストレート          0.433767  0.489510             シンカー   
7         広島           ストレート          0.416643  0.412587             シンカー   
8       日本ハム           ストレート          0.400264  0.580420              特殊球   
9         楽天           ストレート          0.452164  0.468531              特殊球   
10        西武           ストレート          0.440373  0.440559              特殊球   
11        阪神           ストレート          0.440202  0.594406              特殊球   

In [14]:
import sqlite3


con = sqlite3.connect('baseball_analysis.db')


summary.to_sql('team_pitch_summary', con, if_exists='replace', index=False)

con.close()